# 📊 The Precision-Recall (PR) Curve under Class Imbalance

Welcome to the hands-on explanation notebook for the **Precision-Recall (PR) Curve**! In this notebook, we will:
1. Explain why PR Curves are the industry standard for evaluating object detectors like YOLO, rather than ROC Curves.
2. Implement the PR Curve coordinate calculations from scratch using NumPy.
3. Simulate a highly imbalanced dataset (typical of object detection where background is the dominant class).
4. Plot and compare the **ROC Curve** vs. the **PR Curve** on the exact same dataset to illustrate how ROC curves hide false positives under class imbalance.
5. Define **Average Precision (AP)** and **mean Average Precision (mAP)**.

Let's start by importing the necessary libraries.

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
from sklearn.metrics import precision_recall_curve, roc_curve, auc

# Set seed for reproducibility
np.random.seed(42)

## 1. Simulating a Highly Imbalanced Dataset
In object detection, positive bounding boxes are rare compared to negative background predictions.
We simulate:
-   20 actual objects (Class 1).
-   180 background regions (Class 0).
-   Model output scores with some false positives (giving background clutter high confidence).

In [ ]:
y_true = np.concatenate([np.ones(20), np.zeros(180)]).astype(int)

# Bounding box confidence scores:
# Actual objects: mean confidence 0.75
scores_objects = np.random.normal(0.75, 0.15, 20)
# Background noise: mean confidence 0.35, but with some high-scoring false alarms
scores_bg = np.random.normal(0.35, 0.15, 180)

y_scores = np.concatenate([scores_objects, scores_bg])
y_scores = np.clip(y_scores, 0.0, 1.0)

## 2. Calculating the PR Curve from Scratch

Let's write a function to evaluate Precision and Recall at all possible thresholds.
Steps:
1. Sort scores descending.
2. For each score as a threshold, calculate predictions: `y_pred = (scores >= thresh)`.
3. Compute TP, FP, FN.
4. Calculate Precision ($TP / (TP + FP)$) and Recall ($TP / (TP + FN)$).

In [ ]:
def custom_precision_recall_curve(y_true, scores):
    """
    Calculate Precision-Recall coordinates from scratch.
    """
    thresholds = np.sort(scores)[::-1]
    
    precisions = []
    recalls = []
    
    for thresh in thresholds:
        y_pred = (scores >= thresh).astype(int)
        
        TP = np.sum((y_true == 1) & (y_pred == 1))
        FP = np.sum((y_true == 0) & (y_pred == 1))
        FN = np.sum((y_true == 1) & (y_pred == 0))
        
        precision = TP / (TP + FP) if (TP + FP) > 0 else 1.0
        recall = TP / (TP + FN) if (TP + FN) > 0 else 0.0
        
        precisions.append(precision)
        recalls.append(recall)
        
    # Append boundary points
    precisions = np.concatenate([[1.0], precisions])
    recalls = np.concatenate([[0.0], recalls])
    
    return np.array(precisions), np.array(recalls)

# Compute scratch coordinates
prec_scratch, rec_scratch = custom_precision_recall_curve(y_true, y_scores)

# Get sklearn version
prec_sk, rec_sk, _ = precision_recall_curve(y_true, y_scores)

print("Recall matches closely?", np.allclose(np.interp(rec_sk, rec_scratch, rec_scratch), rec_sk))

## 3. ROC Curve vs. PR Curve under Imbalance

Let's plot both curves side-by-side using the same dataset.

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(16, 6))

# Plot ROC Curve
fpr_sk, tpr_sk, _ = roc_curve(y_true, y_scores)
roc_auc = auc(fpr_sk, tpr_sk)

axes[0].plot(fpr_sk, tpr_sk, color='forestgreen', linewidth=3, label=f'ROC (AUC = {roc_auc:.3f})')
axes[0].plot([0, 1], [0, 1], color='gray', linestyle='--', label='Random Guess (AUC = 0.50)')
axes[0].set_xlabel('False Positive Rate (FPR)')
axes[0].set_ylabel('True Positive Rate (TPR / Recall)')
axes[0].set_title('ROC Curve (Deceptively Optimistic)')
axes[0].grid(True, linestyle='--', alpha=0.5)
axes[0].legend(loc='lower right')

# Plot PR Curve
pr_auc = auc(rec_sk, prec_sk)

axes[1].plot(rec_sk, prec_sk, color='darkorange', linewidth=3, label=f'PR Curve (AP/AUC = {pr_auc:.3f})')
axes[1].set_xlabel('Recall')
axes[1].set_ylabel('Precision')
axes[1].set_title('Precision-Recall Curve (Accurate View)')
axes[1].grid(True, linestyle='--', alpha=0.5)
axes[1].legend(loc='lower left')

plt.tight_layout()
plt.show()

Look at the difference!
-   **ROC Curve:** Yields an **AUC of 0.887**. It looks excellent because the False Positive Rate ($FPR = FP / (TN + FP)$) is deflated by the 180 background samples ($TN$).
-   **PR Curve:** Yields an **AP (AUC-PR) of 0.655**. It correctly drops when the model makes false alarms, highlighting that the model is struggling with precision due to false positive detections.

## 💡 Connection to Computer Vision & YOLO
*   **Average Precision (AP):** AP is calculated by finding the area under the PR curve for a class.
*   **mAP@0.5:** The mean Average Precision across all 26 classes, evaluated at an IoU threshold of 0.50. This is the primary metric shown in YOLO training logs.
*   **BoxPR_curve.png:** YOLO generates this PR curve plot for every class during validation. If a class curve dips early, it signals that the model makes too many false alarms for that specific category.